$$\Large\boxed{\text{AME 5202 Deep Learning, Even Semester 2026}}$$

$$\large\text{Theme}: \underline{\text{demonstrating step-by-step the forward and backward propagation of a softmax classifier for a batch of samples}}$$

---

Load essential libraries

---

In [10]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline
import sys
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
import seaborn as sns

---

Mount Google Drive folder if running Google Colab

---

In [6]:
## Mount Google drive folder if running in Colab
if('google.colab' in sys.modules):
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    DIR = '/content/drive/MyDrive/Colab Notebooks/MAHE/MSIS Coursework/EvenSem2026MAHE'
    DATA_DIR = DIR+'/Data/'
else:
    DATA_DIR = 'Data/'

---

**The patient data matrix with output labels and initial weight matrix**

![patient dataset](https://1drv.ms/i/s!AjTcbXuSD3I3hspfrgklysOtJMOjaA?embed=1&width=800)

---



In [7]:
# Patients data matrix
X = torch.tensor([[72, 120, 37.3, 104, 32.5],
                 [85, 130, 37.0, 110, 14],
                 [68, 110, 38.5, 125, 34],
                 [90, 140, 38.0, 130, 26],
                 [84, 132, 38.3, 146, 30],
                 [78, 128, 37.2, 102, 12]], dtype = torch.float64)
print(f'Patient data matrix X:\n {X}') #f-string in Python
print('---------')

# Initial Weights matrix (trainable tensor)
W = torch.tensor([[-0.1, 0.5, 0.3],
                  [0.9, 0.3, 0.5],
                  [-1.5, 0.4, 0.1],
                  [0.1, 0.1, -1.0],
                  [-1.2, 0.5, -0.8]], dtype = torch.float64,
                 requires_grad = True)
print(f'Initial weights matrix:\n {W}')
print('---------')

# Create a 1D-numpy array of output labels (equivalent to a rank-1 tensor in
# PyTorch which itself is equivalent to a vector in pen & paper)
y = np.array(['non-diabetic',
              'diabetic',
              'non-diabetic',
              'pre-diabetic',
              'diabetic',
              'pre-diabetic'])
# Creating a one-hot encoder object
ohe = OneHotEncoder(sparse_output = False)
# Create the one-hot encoded true output labels matrix
Y = torch.tensor(ohe.fit_transform(y.reshape(-1, 1)), dtype = torch.float64)
print(f'One-hot encoded output labels matrix:\n {Y}')
print('---------')

# Standardize the data
sc = StandardScaler() # create a standard scaler object
X_std = torch.tensor(sc.fit_transform(X), dtype = torch.float64)
print(f'The standardized data matrix:\n{X_std}')

Patient data matrix X:
 tensor([[ 72.0000, 120.0000,  37.3000, 104.0000,  32.5000],
        [ 85.0000, 130.0000,  37.0000, 110.0000,  14.0000],
        [ 68.0000, 110.0000,  38.5000, 125.0000,  34.0000],
        [ 90.0000, 140.0000,  38.0000, 130.0000,  26.0000],
        [ 84.0000, 132.0000,  38.3000, 146.0000,  30.0000],
        [ 78.0000, 128.0000,  37.2000, 102.0000,  12.0000]],
       dtype=torch.float64)
---------
Initial weights matrix:
 tensor([[-0.1000,  0.5000,  0.3000],
        [ 0.9000,  0.3000,  0.5000],
        [-1.5000,  0.4000,  0.1000],
        [ 0.1000,  0.1000, -1.0000],
        [-1.2000,  0.5000, -0.8000]], dtype=torch.float64, requires_grad=True)
---------
One-hot encoded output labels matrix:
 tensor([[0., 1., 0.],
        [1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.],
        [1., 0., 0.],
        [0., 0., 1.]], dtype=torch.float64)
---------
The standardized data matrix:
tensor([[-0.9799, -0.7019, -0.7238, -0.9871,  0.8920],
        [ 0.7186,  0.3509, 

---

Assume that there are 4 samples for simplicity. The samples in the batch are represented as the $4\times5$-matrix:

$$\mathbf{X} = \begin{bmatrix}{\mathbf{x}^{(0)}}^\mathrm{T}\\{\mathbf{x}^{(1)}}^\mathrm{T}\\{\mathbf{x}^{(2)}}^\mathrm{T}\\{\mathbf{x}^{(3)}}^\mathrm{T}\end{bmatrix}$$ with one-hot encoded true labels represented as the $4\times3$-matrix $$\mathbf{Y}=\begin{bmatrix}{\mathbf{y}^{(0)}}^\mathrm{T}\\{\mathbf{y}^{(1)}}^\mathrm{T}\\{\mathbf{y}^{(2)}}^\mathrm{T}\\{\mathbf{y}^{(3)}}^\mathrm{T}\end{bmatrix}.$$

The forward propagation on the computer for a generic sample in the batch seen as a $1\times5$-object $\mathbf{x}^\mathrm{T}$ is presented below:

$$\small\begin{align*}
\boxed{\underbrace{\mathbf{x}^\mathrm{T}}_{1\times5}}&\rightarrow\boxed{\underbrace{{\mathbf{z}}^\mathrm{T}}_{1\times 3} = \underbrace{\mathbf{x}^\mathrm{T}}_{1\times5}\underbrace{{\mathbf{W}}}_{5\times3}}\rightarrow\boxed{\underbrace{{\mathbf{a}}^\mathrm{T}}_{1\times3}=\text{softmax}\left(\underbrace{{\mathbf{z}}^\mathrm{T}}_{1\times3}\right)}\rightarrow\boxed{L = \sum\limits_{k=0}^2-y_k\log\left(\hat{y}_k\right)=-\mathbf{y}^\mathrm{T}\log\left(\mathbf{\hat{y}}\right)}.
\end{align*}$$

The forward propagation on pen & paper for the same generic sample seen as a $5$-vector $\mathbf{x}$ is presented below: (note that the weight matrix has the same name $\mathbf{W}$ as above for simplicity even though it should show up as $\mathbf{W}^\mathrm{T}$):

$$\small\begin{align*}
\boxed{\underbrace{\mathbf{x}}_{5\times1}}&\rightarrow\boxed{\underbrace{\mathbf{z}}_{3\times1} = \underbrace{\mathbf{W}}_{3\times5}\underbrace{\mathbf{x}}_{5}}\rightarrow\boxed{\underbrace{\mathbf{a}}_{3\times1}=\text{softmax}\left(\underbrace{\mathbf{z}}_{3\times1}\right)}\rightarrow\boxed{L = \sum\limits_{k=0}^2-y_k\log\left(\hat{y}_k\right)=-\mathbf{y}^\mathrm{T}\log\left(\mathbf{\hat{y}}\right)}.\end{align*}$$

We will derive the update rule for the weights matrix $\mathbf{W}$ using the setup above.


The average crossentropy (CCE) training loss for the batch is:$$\begin{align*}L &=\frac{1}{4}\left[L_0+L_1+L_2+L_3\right]\\&=\frac{1}{4}\left[{\color{yellow}-}{\mathbf{y}^{(0)}}^\mathrm{T}\log\left({\hat{\mathbf{y}}^{(0)}}\right)+{\color{yellow}-}{\mathbf{y}^{(1)}}^\mathrm{T}\log\left({\hat{\mathbf{y}}^{(1)}}\right)+{\color{yellow}-}{\mathbf{y}^{(2)}}^\mathrm{T}\log\left({\hat{\mathbf{y}}^{(2)}}\right)+{\color{yellow}-}{\mathbf{y}^{(3)}}^\mathrm{T}\log\left({\hat{\mathbf{y}}^{(3)}}\right)\right].\end{align*}$$

The computational graph for the samples, each at a time treated as a $5$-vector, in the batch are presented below where the weights matrix has shape $3\times 5$:

$$\hspace{1.5in}\begin{align*}L_0\\{\color{yellow}\downarrow}\\ \hat{\mathbf{y}}^{(0)} &= \mathbf{a}^{(0)}\\{\color{yellow}\downarrow}\\\mathbf{z}^{(0)}\\{\color{yellow}\downarrow}\\\mathbf{W}\end{align*}\hspace{0.25in}\begin{align*}L_1\\{\color{yellow}\downarrow}\\ \hat{\mathbf{y}}^{(1)} &= \mathbf{a}^{(1)}\\{\color{yellow}\downarrow}\\\mathbf{z}^{(1)}\\{\color{yellow}\downarrow}\\\mathbf{W}\end{align*}\qquad\begin{align*} L_{2}\\{\color{yellow}\downarrow}\\ \hat{\mathbf{y}}^{(2)} &= \mathbf{a}^{(2)}\\{\color{yellow}\downarrow}\\\mathbf{z}^{(2)}\\{\color{yellow}\downarrow}\\\mathbf{W}\end{align*}\qquad\begin{align*} L_{3}\\{\color{yellow}\downarrow}\\ \hat{\mathbf{y}}^{(3)} &= \mathbf{a}^{(3)}\\{\color{yellow}\downarrow}\\\mathbf{z}^{(3)}\\{\color{yellow}\downarrow}\\\mathbf{W}\end{align*}\hspace{5in}$$

The forward and backward propagation showing the gradient flow for a generic sample is shown below:

![](https://1drv.ms/i/c/37720f927b6ddc34/IQRbdiSbwjCRQK5VjcRO7PGfAYtvNPKCyciTCOjpvxkE0Y8?width=660)

The gradient of the average training loss w.r.t. the weights is:
$$\small\begin{align*}\Rightarrow \nabla_\mathbf{W}(L) &=\frac{1}{4}\left[\nabla_\mathbf{W}(L_0)+\nabla_\mathbf{W}(L_1)+\nabla_\mathbf{W}(L_2)+\nabla_\mathbf{W}(L_{3})\right]\end{align*}$$
which by chain rule can be written as:

$$\small\begin{align*}\Rightarrow \nabla_\mathbf{W}(L) &=\frac{1}{4}\sum_{i=0}^{3}\left[\nabla_\mathbf{W}\left(\mathbf{z}^{(i)}\right) \times\nabla_{\mathbf{z}^{(i)}}\left({\mathbf{a}}^{(i)}\right)\times\nabla_{\hat{\mathbf{y}}^{(i)}}(L_i)\right]\\&=\frac{1}{4}\sum_{i=0}^{3}\left[\nabla_\mathbf{W}\left(\mathbf{W}{\mathbf{x}^{(i)}}\right) \times\nabla_{\mathbf{z}^{(i)}}\left(\text{softmax}\left({\mathbf{z}}^{(i)}\right)\right)\times\nabla_{\hat{\mathbf{y}}^{(i)}}\left(-{\mathbf{y}^{(i)}}^\mathrm{T}\log\left(\hat{\mathbf{y}}^{(i)}\right)\right)\right],\end{align*}$$
which can be written as

$$\small\begin{align*}\nabla_{\mathbf{W}}(L) &= \dfrac{1}{4}\displaystyle\sum_{i=0}^{3}\underbrace{\begin{bmatrix}\boxed{{\mathbf{x}^{(i)}}\quad\pmb{0}\quad\pmb{0}}&&&&\\\\&\boxed{\pmb{0}\quad{\mathbf{x}^{(i)}}\quad\pmb{0}}&&&\\&&&\boxed{\pmb{0}\quad\pmb{0}\quad{\mathbf{x}^{(i)}}}&\end{bmatrix}}_{\color{cyan}{\nabla_\mathbf{W}\left(\mathbf{z}^{(i)}\right)=\nabla_\mathbf{W}\left(\mathbf{W}{\mathbf{x}^{(i)}}\right):\ 3\times5\times3}}\underbrace{\begin{bmatrix}a^{(i)}_0 (1 - a^{(i)}_0) & -a^{(i)}_0 a^{(i)}_1 & -a^{(i)}_0 a^{(i)}_2\\-a^{(i)}_1 a^{(i)}_0 & a^{(i)}_1 (1 - a^{(i)}_1) & -a^{(i)}_1 a^{(i)}_2\\-a^{(i)}_2 a^{(i)}_0 & -a^{(i)}_2 a^{(i)}_1 & a^{(i)}_2 (1 - a^{(i)}_2)\end{bmatrix}}_{\color{cyan}{\nabla_{\mathbf{z}^{(i)}}\left({\mathbf{a}}^{(i)}\right) = \nabla_{\mathbf{z}^{(i)}}\left(\text{softmax}\left({\mathbf{z}}^{(i)}\right)\right):\ 3\times3}}\underbrace{\begin{bmatrix}-\frac{y^{(i)}_0}{\hat{y}^{(i)}_0}\\-\frac{y^{(i)}_1}{\hat{y}^{(i)}_1}\\-\frac{y^{(i)}_2}{\hat{y}^{(i)}_2}\end{bmatrix}}_{\color{cyan}{\nabla_{\hat{\mathbf{y}}^{(i)}}(L_i)=\nabla_{\hat{\mathbf{y}}^{(i)}}\left(-{\mathbf{y}^{(i)}}^\mathrm{T}\log\left(\hat{\mathbf{y}}^{(i)}\right)\right):\ 3\times1}}\end{align*}$$

where the gradient terms inside the summation can be calculated and written as

$$\begin{align*}\nabla_{\mathbf{W}}(L) &=\dfrac{1}{4}\displaystyle\sum_{i=0}^{3}\underbrace{\begin{bmatrix}a^{(i)}_0 (1 - a^{(i)}_0) & -a^{(i)}_0 a^{(i)}_1 & -a^{(i)}_0 a^{(i)}_2\\-a^{(i)}_1 a^{(i)}_0 & a^{(i)}_1 (1 - a^{(i)}_1) & -a^{(i)}_1 a^{(i)}_2\\-a^{(i)}_2 a^{(i)}_0 & -a^{(i)}_2 a^{(i)}_1 & a^{(i)}_2 (1 - a^{(i)}_2)\end{bmatrix}}_{\color{cyan}{=\left(\mathbf{I}-{\mathbf{a}^{(i)}}^\mathrm{T}\right)\otimes\mathbf{a}^{(i)}}}\underbrace{\begin{bmatrix}-\frac{y^{(i)}_0}{\hat{y}^{(i)}_0}\\-\frac{y^{(i)}_1}{\hat{y}^{(i)}_1}\\-\frac{y^{(i)}_2}{\hat{y}^{(i)}_2}\end{bmatrix}}_{\color{cyan}{=-\frac{\mathbf{y}^{(i)}}{\hat{\mathbf{y}}^{(i)}}}}{\mathbf{x}^{(i)}}^\mathrm{T}.\end{align*}$$

We can write the gradient in the following way for efficient coding purposes: $$\nabla_\mathbf{W}(L) = \frac{1}{4}\sum_{i=0}^{3}\left[\left(\mathbf{I}-{\mathbf{a}^{(i)}}^\mathrm{T}\right)\otimes\mathbf{a}^{(i)}\right]\left[-\frac{\mathbf{y}^{(i)}}{\hat{\mathbf{y}}^{(i)}}\right]{\mathbf{x}^{(i)}}^\mathrm{T}.$$


It can be seen that the gradient object has shape $(3\times 3)\times(3\times1)\times(1\times5)=3\times5,$ which is the same shape as the weights matrix $\mathbf{W}.$ However, our derivation here assumed that the samples are seen as column vectors of the data matrix. The original data matrix has the samples arranged as rows which corresponded to the weights matrix of shape $5\times3.$ In order to get the gradient w.r.t. that weights matrix, we have to transpose this expression resulting in the update $$\nabla_\mathbf{W}(L) = \frac{1}{4}\sum_{i=0}^{3}\underbrace{\mathbf{x}^{(i)}}_{\color{yellow}{5\times1}}\underbrace{\underbrace{\left[-\frac{{\mathbf{y}^{(i)}}^\mathrm{T}}{{\hat{\mathbf{y}}^{(i)}}^\mathrm{T}}\right]}_{\color{magenta}{\text{output side gradient of softmax layer: }1\times3}}\underbrace{\left[\left(\mathbf{I}-{\mathbf{a}^{(i)}}\right)\otimes{\mathbf{a}^{(i)}}^\mathrm{T}\right]}_{\color{magenta}{\text{local gradient of softmax layer: }3\times3}}}_{\color{yellow}{\text{output side gradient of dense layer: }1\times3}}.$$

Now we perform the forward propagation for all the samples in the training batch efficiently.


---

In [8]:
# Raw scores
Z = X_std @ W # shape 6x3
print(f'Raw scores matrix with shape {tuple(Z.shape)}:\n{Z}')

# Softmax-activated scores
A = F.softmax(Z, dim = -1) # shape 6x3
print(f'Softmax-activated scores matrix with shape {tuple(A.shape)}:\n{A}')
# The following also works
#softmax = torch.nn.Softmax(dim = -1)
#A = softmax(Z)

# Calculate the average training loss
L = torch.mean(-torch.log(torch.sum(Y*A, dim = -1)))
print(f'Average training loss = {L}')
# The following also work
#L = F.cross_entropy(Z, Y)
#cce = torch.nn.CrossEntropyLoss()
#L = cce(Z, Y)


Raw scores matrix with shape (6, 3):
tensor([[-0.6171, -0.6427, -0.4438],
        [ 3.5357, -0.7126,  1.8614],
        [-4.7127, -0.1660, -2.3940],
        [ 0.2821,  1.4427,  0.3789],
        [-1.6298,  1.3386, -1.6126],
        [ 3.1418, -1.2601,  2.2101]], dtype=torch.float64,
       grad_fn=<MmBackward0>)
Softmax-activated scores matrix with shape (6, 3):
tensor([[0.3161, 0.3081, 0.3759],
        [0.8321, 0.0119, 0.1560],
        [0.0095, 0.8942, 0.0963],
        [0.1889, 0.6030, 0.2081],
        [0.0466, 0.9061, 0.0474],
        [0.7112, 0.0087, 0.2801]], dtype=torch.float64,
       grad_fn=<SoftmaxBackward0>)
Average training loss = 1.2303940464309857


---

Backward propagation to calculate the gradient of the loss w.r.t. the weights using a for-loop:

$$\nabla_\mathbf{W}(L) = \frac{1}{6}\sum_{i=0}^{5}\underbrace{\mathbf{x}^{(i)}}_{\color{yellow}{5\times1}}\underbrace{\underbrace{\left[-\frac{{\mathbf{y}^{(i)}}^\mathrm{T}}{{\hat{\mathbf{y}}^{(i)}}^\mathrm{T}}\right]}_{\color{magenta}{\text{output side gradient of softmax layer: }1\times3}}\underbrace{\left[\left(\mathbf{I}-{\mathbf{a}^{(i)}}\right)\otimes{\mathbf{a}^{(i)}}^\mathrm{T}\right]}_{\color{magenta}{\text{local gradient of softmax layer: }3\times3}}}_{\color{yellow}{\text{output side gradient of dense layer: }1\times3}}.$$

---

In [12]:
grad = torch.zeros(W.shape, dtype = torch.float64)
I = torch.eye(3)

start_time = time.process_time()

for i in range(X_std.shape[0]):
    loss_gradient = (-Y[i]/A[i]).unsqueeze(-2)
    softmax_local_gradient = (I - A[i].unsqueeze(-1))*(A[i].unsqueeze(-2))
    grad += (X_std[i].unsqueeze(-1)) @ (loss_gradient @ softmax_local_gradient)

grad = grad / X_std.shape[0]

end_time = time.process_time()

print(f'The gradient w.r.t. the weights = \n{grad}')
print(f'Execution time = {end_time-start_time:0.4f}s')

The gradient w.r.t. the weights = 
tensor([[-0.1476,  0.3673, -0.2197],
        [-0.0780,  0.3386, -0.2607],
        [-0.2531,  0.2582, -0.0051],
        [-0.4137,  0.4269, -0.0132],
        [-0.1822, -0.0205,  0.2027]], dtype=torch.float64,
       grad_fn=<DivBackward0>)
Execution time = 0.0312s


---

Backward propagation for all the samples in the batch efficiently using einsum (watch this short and excellent video on einsum [video](https://www.youtube.com/watch?v=pkVwUVEHmfI)):

![einsum](https://1drv.ms/i/c/37720f927b6ddc34/IQRAlj4G7t54TZKd74qGYB2DAR6gty7du7V6kwgqANGkWfw?width=660)

---

In [14]:
start_time = time.process_time()

# Loss layer
grad = -Y / A # shape 6 x 3

# Softmax layer
softmax_local_gradient = (I-A.unsqueeze(-1))*(A.unsqueeze(-2)) # 6 x 3 x 3
grad = torch.einsum('ij, ijk -> ik', grad, softmax_local_gradient) # 6 x 3

# Dense layer
grad = torch.einsum('ij, ik -> jk', X_std, grad) # shape 5x3

grad = grad/X_std.shape[0]

end_time = time.process_time()

print(f'The gradient w.r.t. the weights = \n{grad}')
print(f'Execution time = {end_time-start_time:0.4f}s')

The gradient w.r.t. the weights = 
tensor([[-0.1476,  0.3673, -0.2197],
        [-0.0780,  0.3386, -0.2607],
        [-0.2531,  0.2582, -0.0051],
        [-0.4137,  0.4269, -0.0132],
        [-0.1822, -0.0205,  0.2027]], dtype=torch.float64,
       grad_fn=<DivBackward0>)
Execution time = 0.0156s


---

Update weights

---